# From satellite and radar to convection-initiation forecasts
## Read the paper’s results, then explore a real case

**Spatially Explicit Deep Learning Nowcasting of Convection Initiation Using GOES and MRMS**

Marc Crampe<sup>1,2,3,*</sup> · Corey K. Potvin<sup>5,6</sup> · Jacob T. Carlin<sup>5</sup> · Jonathan J. Gourley<sup>5,6</sup> · Derek Stratman<sup>4,5</sup> · Christopher A. Kerr<sup>4,5</sup> · Said A. Mejia Manrique<sup>1,2,3</sup> · Reza Khanbilvardi<sup>1,2,3</sup>

<details>
<summary><strong>Author affiliations</strong></summary>

1. Department of Civil Engineering, The City College of New York, City University of New York, New York, NY, USA.
2. CUNY CREST Institute, The City College of New York, New York, NY, USA.
3. United Nations University (UNU) Hub on Remote-Sensing and Sustainable Innovations for Resilient Urban Systems (R-SIRUS) at the City College of New York, United Nations University Institute for Water, Environment and Health (UNU-INWEH), New York, USA.
4. Cooperative Institute for Severe and High-Impact Weather Research and Operations, The University of Oklahoma, Norman, OK, USA.
5. NOAA/OAR National Severe Storms Laboratory, Norman, OK, USA.
6. School of Meteorology, The University of Oklahoma, Norman, OK, USA.

</details>

[Repository](https://github.com/Marccrampe/goes-mrms-ci-nowcasting) · [Correspondence](mailto:mcrampe@ccny.cuny.edu)

The residual U-Net combines GOES-East infrared observations with MRMS-derived radar features
to forecast CI occurrence in two successive windows. This notebook connects the manuscript’s
aggregate results to the individual objects behind a forecast.

**Two independent demonstrations:**

1. Explore the paper’s **25 July 2025, 19:00 UTC** case and change verification criteria.
   Only its **0–30-min** forecast is archived.
2. Optionally run the released model on the separate miniature **TEST** sample, producing both forecast windows.

The case explorer needs **CPU only**, no model weights and no raw-data downloads.
It checks the archived files and original classifications before exposing the controls.


## 1 · What is the model trained to predict?

The target representation changes how CI locations enter training. The supplied figure compares
six soft/hard representations; the bottom panels report strict one-to-one TEST CSI after
threshold selection on VALIDATION. The released final model uses **hard +2 model-grid pixels**.

The expanded training target is **not** the observed verification footprint. In the interactive
case below, verification always starts from the canonical, uninflated CI35 birth cores.


![Manuscript figure](https://raw.githubusercontent.com/Marccrampe/goes-mrms-ci-nowcasting/main/docs/figures/Fig04_V1_strict_one_to_one_VALopt_TEST.png)


## 2 · What counts as a successful forecast?

Strict CI35 verification and weaker persistent-radar-growth diagnostics answer different questions.
The supplied model-comparison figure illustrates their different scores. Its orange bars use
**≥25 dBZ, 10-min persistence, one native MRMS birth pixel and 15-km association**.

These aggregate manuscript results are included as context. They are **not recomputed from this single case**.
The interactive case uses the archived Fig. 11 diagnostic implementation, whose accounting is
specified below; do not assume that every CSI* figure uses identical reassociation/FN rules.


![Manuscript figure](https://raw.githubusercontent.com/Marccrampe/goes-mrms-ci-nowcasting/main/docs/figures/Fig09_model_definition_CSI_P10.png)


### A separate question: one-to-one matching competition

One forecast object can lie near several observed CI35 initiations, while conventional matching
assigns it to only one. The supplied example shows why additional observations may remain FN.
Recovering those competition FN is a **separate diagnostic** from accepting a forecast on the
basis of weaker radar growth. The controls below do not combine the two operations.


![Manuscript figure](https://raw.githubusercontent.com/Marccrampe/goes-mrms-ci-nowcasting/main/docs/figures/Supplementary_B_real_competition.png)


## 3 · Start the case explorer

**Colab:** run the next cell. It downloads a fresh copy of the public GitHub repository,
including the committed case data. No manual upload, checkpoint or GPU is needed.
The case assets must be committed at their original paths on the `main` branch.

**Local Jupyter:** extract the ZIP and open this notebook from its `notebooks/` directory.


In [ ]:
from pathlib import Path
import sys, subprocess, tempfile, os, json, urllib.request, zipfile, io, importlib

REPOSITORY = "Marccrampe/goes-mrms-ci-nowcasting"
REF = "main"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("1/4 Downloading the public repository and case data...", flush=True)
    archive_bytes = None
    errors = []
    urls = [f"https://api.github.com/repos/{REPOSITORY}/zipball/{REF}",
            f"https://codeload.github.com/{REPOSITORY}/zip/refs/heads/{REF}"]
    for url in urls:
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "GOES-MRMS-Colab"})
            with urllib.request.urlopen(request, timeout=120) as response:
                archive_bytes = response.read(100 * 1024 * 1024 + 1)
            break
        except Exception as exc:
            errors.append(str(exc))
    if archive_bytes is None:
        raise RuntimeError("GitHub download failed. Check that the repository is public and retry. "
                           + " | ".join(errors))
    if len(archive_bytes) > 100 * 1024 * 1024:
        raise RuntimeError("Unexpected archive size: model weights must remain separate Release assets.")
    checkout = Path(tempfile.mkdtemp(prefix="ci-colab-"))
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        for member in archive.infolist():
            if not (checkout / member.filename).resolve().is_relative_to(checkout.resolve()):
                raise RuntimeError("Unsafe archive member")
            if ((member.external_attr >> 16) & 0o170000) == 0o120000:
                raise RuntimeError("Unexpected symlink in source archive")
        archive.extractall(checkout)
    roots = [p for p in checkout.iterdir() if (p / "tools/object_demo.py").is_file()]
    if len(roots) != 1:
        raise RuntimeError("The repository archive must contain tools/object_demo.py at its root.")
    root = roots[0]
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    root = next((p for p in candidates if (p / "tools/object_demo.py").is_file()), None)
    if root is None:
        raise RuntimeError("Open this notebook from inside the extracted repository.")

print("2/4 Checking runtime dependencies...", flush=True)
required_imports = {"numpy": "numpy>=1.26,<3", "scipy": "scipy>=1.13,<2",
    "pandas": "pandas>=2.2,<3", "matplotlib": "matplotlib>=3.9,<4",
    "ipywidgets": "ipywidgets>=8,<9", "IPython": "ipython>=8"}
missing = []
for module, requirement in required_imports.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

print("3/4 Checking the complete case-file inventory...", flush=True)
case_root = root / "examples/fig11_case"
manifest = case_root / "manifest.json"
if not manifest.is_file():
    raise RuntimeError("Missing examples/fig11_case/manifest.json in the public repository.")
for row in json.loads(manifest.read_text()):
    path = (case_root / row["exported"]).resolve()
    if not path.is_relative_to(case_root.resolve()) or not path.is_file():
        raise RuntimeError("Missing or invalid case asset: " + row["exported"])
    with path.open("rb") as stream:
        if stream.read(100).startswith(b"version https://git-lfs.github.com/spec/"):
            raise RuntimeError("Case asset is a Git LFS pointer: " + row["exported"])

sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
for name in ("object_demo", "object_demo_kernels", "demo_support"):
    sys.modules.pop(name, None)
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass
print("4/4 Ready — run the next cell to verify the original case.", flush=True)
print("Python:", sys.version.split()[0], "| case files:", root)


### Verify the reference result first

The case contains full-domain probabilities, canonical CI35 core indices, original decisions,
projected radar scans and native-radar lineage profiles. No training target is used as truth.

At the original raw threshold **0.05**, strict **5-km** matching and a minimum **2 forecast pixels**,
the reference is **TP 7 / FP 10 / FN 9**. The original ≥25-dBZ/P5/15-km diagnostic accepts
**8 additional forecasts** and leaves **2 unmatched forecasts**.

The exported config’s diagnostic description disagrees with the saved case: the config says
30 dBZ/P10, whereas the saved decisions say **25 dBZ/P5**. The notebook reproduces and validates
the **saved decisions**, then labels every exploratory setting explicitly.


In [ ]:
from object_demo import CaseStudy, launch, radar_player
from IPython.display import display
case = CaseStudy(root/'examples/fig11_case')
display(case.evaluate()['metrics'].round(6))
print('PASS: archived hashes, original TP/FP/FN, matched object IDs and growth associations reproduced.')
print('Period:', case.score['period_id'], '| forecast: (0,30] min')


## 4 · Change the criteria and explore the maps

- **Probability threshold** changes the connected forecast objects; these changes are exploratory.
- **Persistence** filters the observed ≥25-dBZ lineages using their saved temporal profiles.
- **Growth radius** controls association for the weaker-growth diagnostic. Strict CI35 remains at **5 km**.
- **Native birth area** filters radar candidates; forecast objects still require two model-grid pixels.
- **Display extent** changes only the view. Matching and counts always use the full **212 × 230** domain.

**Green:** accepted forecasts. **Red:** unmatched forecasts. **Blue circles:** strict matched CI35.
**Blue crosses:** strict CI35 FN, including competition FN. **Orange circles:** accepted ≥25-dBZ growth candidates.

The original growth rule additionally requires novelty, at least three contiguous scans,
intensity or native-area growth, valid coverage and topology checks. Persistence alone is insufficient.
Only ≥25-dBZ candidate lineages are archived, so no unsupported 30/35-dBZ selector is offered.

Select **Show forecast IDs** to identify individual forecast objects directly on the maps.

**Uncalibrated output:** raw sigmoid values are model scores, not calibrated CI occurrence frequencies. No isotonic mapping is applied in this explorer.


In [ ]:
state = launch(case)


## 5 · Follow the radar through the forecast window

Play or step through every available radar scan in (19:00, 19:30] UTC. The optional white
outlines remain the **original p = 0.05 forecast objects**, independently of the exploratory controls above.
The radar imagery is projected with the original maximum-reflectivity mapper, not linear interpolation.


In [ ]:
radar_player(case)


## 6 · Read the accounting

Let strict counts be TP, FP and FN, and let G be the number of eligible strict-FP forecasts
accepted by the radar-growth diagnostic. Strict TP, strict FN and forecast geometry are retained:

| Quantity | Strict | Fig. 11 growth diagnostic |
|---|---|---|
| Accepted forecasts | TP | TP + G |
| Unmatched forecasts | FP | FP − G |
| Strict observed FN | FN | FN |
| CSI / CSI* | TP / (TP + FP + FN) | (TP + G) / (TP + FP + FN) |
| POD / POD* | TP / (TP + FN) | (TP + G) / (TP + G + FN) |
| FAR / FAR* | FP / (TP + FP) | (FP − G) / (TP + FP) |

Duplicate forecasts near strict observations are excluded from growth promotion. Matching is one-to-one
between eligible growth candidates and eligible forecasts. **Competition FN are not recovered here.**
The diagnostic scores are not conventional CI35 event scores or the separate association-based CSI.

At the original settings, the counts yield CSI = 7/26 and CSI* = 15/26. The agreement with the
saved figure is a reproducibility check; these single-case scores do not replace the aggregate evaluation.

Use **Export current tables + settings** above to save the current metrics, object ledger and protocol settings.


## 7 · Optional: run the released neural network

The case explorer above uses the original full-domain saved forecast. It does not run inference.
The repository’s separate miniature TEST sample can exercise the released checkpoint and produce
both 0–30- and 30–60-min outputs. This optional section downloads ~103 MiB of weights and requires
the complete repository; it does not replace the saved Figure 11 forecast.


In [ ]:
RUN_MODEL_DEMO = False  # Change to True to run the separate miniature TEST inference.
if RUN_MODEL_DEMO:
    repo = next((p for p in [root, Path('/content/goes-mrms-ci-nowcasting')]
                 if (p/'models/inference.py').exists() and (p/'tools/demo_support.py').exists()), None)
    if repo is None:
        raise RuntimeError('This optional section needs the complete GitHub repository.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)+'[demo,model]'])
    sys.path.insert(0, str(repo/'tools'))
    from demo_support import run_inference, show_demo
    run_inference(repo)
    import ipywidgets as widgets
    display(widgets.interactive(show_demo, root=widgets.fixed(repo),
        window=widgets.Dropdown(options=[('0–30 min',0), ('30–60 min',1)], description='Window'),
        history=widgets.Dropdown(options=[('−30 min',0), ('−20 min',1), ('−10 min',2), ('T0',3)], value=3),
        display_max=widgets.Dropdown(options=[.1,.25,.5,1.],value=.5,description='Colour max')))
else:
    print('Optional inference skipped. The full case explorer needs no checkpoint or GPU.')


### Provenance and limits

- Source/export checksums: `examples/fig11_case/manifest.json`.
- Numerical functions and source locations: `tools/object_kernel_sources.json`.
- Original scores, associations and object ledger: `examples/fig11_case/verification/`.
- Missing assets and available intervals: `examples/fig11_case/MISSING.md` and `README.md`.
- The supplementary competition image is a **different case**, shown only to explain that mechanism.
- Images above are author-supplied manuscript figures; their aggregate results are static context.
- Only one forecast window and only ≥25-dBZ candidate tracks are supplied for the interactive case.
- The raw output is not isotonic calibrated. The saved case has C13 history, not all seven ABI bands.

**Correspondence:** Marc Crampe · mcrampe@ccny.cuny.edu
